### FIJI OPERATIONS before loading into the notebook
1. Draw rectangle of the region of interest (CA1, CA3 & DG). Make sure that you rotate the image, because the "Plot Profiles" function, takes the profile from left to right and the average from top to bottom.
2. Analyze -> Plot Profiles
3. List -> save as .csv file

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
PROTEIN_NAME = "GPR37L1"
COLOR        = "#e28d21"
SHADE_COLOR  = "#f8e2ca" 
FILES = {
    "CA1": [f"fluor-intensity-data/{PROTEIN_NAME}_CA1_Exp47.csv", f"fluor-intensity-data/{PROTEIN_NAME}_CA1_Exp48.csv", f"fluor-intensity-data/{PROTEIN_NAME}_CA1_Exp51.csv"],
    "DG":  [f"fluor-intensity-data/{PROTEIN_NAME}_DG_Exp47.csv",  f"fluor-intensity-data/{PROTEIN_NAME}_DG_Exp48.csv",  f"fluor-intensity-data/{PROTEIN_NAME}_DG_Exp51.csv"],
    "CA3": [f"fluor-intensity-data/{PROTEIN_NAME}_CA3_Exp47.csv", f"fluor-intensity-data/{PROTEIN_NAME}_CA3_Exp48.csv", f"fluor-intensity-data/{PROTEIN_NAME}_CA3_Exp51.csv"],
}
 
REGIONS = {
    "CA1": {"boundaries": [80, 160, 350],      "labels": ["SO", "SP", "SR", "SLM"]},
    "DG":  {"boundaries": [30, 55, 90, 120],  "labels": ["ML", "GCL", "Hilus", "GCL", "ML"]},
    "CA3": {"boundaries": [30, 55, 100],       "labels": ["SO", "SP", "SL", "SR"]},
}

SG_WINDOW  = 41  

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 6), sharey=False)
fig.suptitle(PROTEIN_NAME, fontsize=10)
 
for ax, (region, files) in zip(axes, FILES.items()):
 
    profiles = [pd.read_csv(f).iloc[:, 1] for f in files]
    distance = pd.read_csv(files[0]).iloc[:, 0]
 
    smoothed = np.array([savgol_filter(p, window_length=SG_WINDOW, polyorder=3) for p in profiles])
    mean     = smoothed.mean(axis=0)
    sem      = smoothed.std(axis=0) / np.sqrt(len(profiles))
 
    norm     = lambda x: (x - mean.min()) / (mean.max() - mean.min())
 
    ax.fill_betweenx(distance, norm(mean - sem), norm(mean + sem), color=SHADE_COLOR, alpha=0.6, linewidth=0)
    ax.plot(norm(mean), distance, color=COLOR, lw=2)
 
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_title(region, fontsize=9)
    ax.set_xlabel("Normalized intensity")
    ax.spines[["top", "right"]].set_visible(False)
 
    boundaries = REGIONS[region]["boundaries"]
    labels     = REGIONS[region]["labels"]
    edges      = [distance.min()] + boundaries + [distance.max()]
 
    for i, label in enumerate(labels):
        ax.text(1.03, (edges[i] + edges[i+1]) / 2, label,
                transform=ax.get_yaxis_transform(), va="center", fontsize=8)
 
axes[0].set_ylabel("Distance (microns)")
 
plt.tight_layout()
plt.savefig(f"{PROTEIN_NAME}_intensity_profile.svg", format="svg", bbox_inches="tight")
 